# Step 5: Deploy automatically when a model is approved

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.3.2</strong> and the <strong>SageMaker Python SDK version 3.20.0</strong>.
</div>

Notebook 04 deployed the approved model, but *you* were the trigger: a cell read the registry, checked the version was `production` and `Approved`, and then created the endpoint. Notebooks 02 and 03 both left the same follow-up on the table: **start the deployment from the `Approved` event, so that the governance officer's decision is the only thing that moves a model to production.**

That is what you build here. When the governance officer sets `ModelApprovalStatus = Approved` on a version, SageMaker emits a `Model Package State Change` event, an EventBridge rule picks out the `Approved` ones for the workshop's model package groups, and a Lambda function deploys the version to a real-time endpoint with data capture enabled. The first approval creates the endpoint; every later approval updates it in place. Nobody runs a deployment cell any more.

||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI processing and training jobs ||
|2b.|Automate model governance with EventBridge and Model Cards ||
|3. |Operationalize with an ML pipeline and model registry ||
|4. |Add model deployment and monitoring ||
|5. |Deploy automatically on approval |**<<<< YOU ARE HERE**|

## What you build

```
 Governance officer              SageMaker                  Amazon EventBridge             AWS Lambda                        SageMaker AI
 UpdateModelPackage              Model Registry             rule: Model Package State      deployment automation             real-time endpoint
 (ModelApprovalStatus=Approved)  emits "Model Package  ---> Change, status = Approved, --> re-reads the registry,       ---> created on the first
 notebook 01 / 03, Studio UI     State Change" event        group name prefix              creates Model + EndpointConfig,   approval, updated in
                                                                                           creates or updates the endpoint   place afterwards
```

- The **event is only a trigger**. The function re-reads the version with `DescribeModelPackage` and deploys it only if it is `Approved`, in the `production` lifecycle stage, and carries an inference specification. A replayed or stale event cannot deploy the wrong thing.
- **One stable endpoint.** The endpoint name never changes; `UpdateEndpoint` performs a blue/green switch to the newly approved version, so callers are not affected. Re-approving an earlier version is a rollback.
- **Data capture stays on**, with the same `DataCaptureConfig` notebook 04 used, so the monitoring built there keeps working across deployments.
- **Idempotent.** SageMaker delivers duplicate events, and the model card automation from notebook 02 updates the package right after the approval, which produces one more `Approved` event. The function checks what the endpoint already serves (or is switching to) and exits without a write when nothing needs to change.

The function's code is in [`deployment_automation/lambda_function.py`](deployment_automation/lambda_function.py); its least-privilege policy in [`deployment_automation/lambda_role_policy.json`](deployment_automation/lambda_role_policy.json).

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/aim-408-end-to-end-modelops-on-sagemaker-ai
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the top

</div>

In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

## Setup

Restore the variables the earlier notebooks stored: the AWS configuration and the governance-officer role from notebook 00, the registered model from notebooks 01 and 03, and the test set location and feature names used to check the endpoint once it is up.

In [ ]:
%store -r

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

required = ["registered_model_name", "test_s3_url", "feature_names", "governance_officer_role_arn"]
missing = [name for name in required if name not in dir()]
if missing:
    raise NameError(
        f"Missing stored variables {missing}. Run notebooks 00, 01 and 03 first; they %store these values."
    )

In [ ]:
import io
import json
import os
import time
import zipfile
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from sklearn.metrics import roc_auc_score

# V3: Core imports
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.s3 import S3Downloader
from IPython.display import HTML, display

# Approval-gate helpers shared with notebooks 01, 03 and 04.
from governance.approval_gate import (
    GOVERNANCE_OFFICER,
    approve_for_deployment,
    as_iam_policy_document,
    assume_persona,
    latest_model_package,
    lifecycle_summary,
    print_lifecycle,
)

session = Session()
sm_role = get_execution_role()
region = session.boto_region_name
account_id = session.account_id()

sm_client = boto3.client("sagemaker")
runtime_client = boto3.client("sagemaker-runtime")
iam_client = boto3.client("iam")
lambda_client = boto3.client("lambda")
events_client = boto3.client("events")
logs_client = boto3.client("logs")

## Set constants

The endpoint gets a **stable name** on purpose: it is the one address applications call, and the automation updates what is behind it. Everything the function creates for a deployment (the `Model` and `EndpointConfig`) is named after the endpoint and the version, which is also what the IAM policy of the function is scoped to.

In [ ]:
project = "from-idea-to-prod"

# The rule matches on this *prefix*, so it also covers the model package groups other
# runs of the workshop create on other days (the auto-sync appends a hash suffix).
model_package_group_prefix = f"{project}-model-"

# These four names must match the DeploymentLambdaFunctionName, DeploymentLambdaRoleName,
# DeploymentRuleName and DeploymentEndpointName parameters of the workshop stack
# (static/cfnSageMakerProjectsRoles.yaml): the role is created there, and the Studio
# execution role's grants are scoped to exactly these names.
deployment_lambda_name = f"{project}-deployment-automation"
deployment_lambda_role_name = f"{project}-deployment-automation-role"
deployment_rule_name = f"{project}-deploy-on-approval"

# The single production endpoint. Created by the first approval, updated by the next ones.
deployment_endpoint_name = f"{project}-endpoint"
inference_instance_type = "ml.m5.xlarge"
# Same capture prefix as notebook 04, so its monitoring code reads this endpoint too.
capture_s3_uri = f"s3://{bucket_name}/{bucket_prefix}/data-capture"

%store deployment_lambda_name
%store deployment_rule_name
%store deployment_endpoint_name

print(f"MLflow registered model:  {registered_model_name}")
print(f"Rule matches groups:      {model_package_group_prefix}*")
print(f"Lambda function:          {deployment_lambda_name}")
print(f"Lambda execution role:    {deployment_lambda_role_name}")
print(f"EventBridge rule:         {deployment_rule_name}")
print(f"Endpoint:                 {deployment_endpoint_name} ({inference_instance_type})")
print(f"Data capture:             {capture_s3_uri}/{deployment_endpoint_name}/AllTraffic/")

<div class="alert alert-warning">
<b>Permissions.</b> Like notebook 02, this notebook deploys an automation, so besides the SageMaker permissions it needs <code>iam:GetRole</code>, <code>iam:ListRolePolicies</code>, <code>iam:GetRolePolicy</code>, <code>iam:PassRole</code> (to <code>lambda.amazonaws.com</code>), <code>lambda:CreateFunction</code>, <code>lambda:UpdateFunctionCode</code>, <code>lambda:UpdateFunctionConfiguration</code>, <code>lambda:GetFunction</code>, <code>lambda:AddPermission</code>, <code>lambda:InvokeFunction</code>, <code>lambda:TagResource</code>, <code>events:PutRule</code>, <code>events:PutTargets</code>, <code>events:DescribeRule</code>, <code>events:DisableRule</code>, <code>events:TagResource</code>, and <code>logs:FilterLogEvents</code> to read the function's logs from here. It also assumes the governance-officer role, exactly as notebooks 01 and 03 do.
<br><br>
The workshop CloudFormation stack (<code>static/cfnSageMakerProjectsRoles.yaml</code>) grants all of that to the Studio execution role through the <code>ModelCardAutomationAccessPolicy</code> managed policy, scoped to the exact function, rule and endpoint names set above; if you rename them here, update the <code>Deployment*</code> parameters of that template too. What the stack deliberately does <em>not</em> grant is <code>iam:CreateRole</code>, which is why the Lambda execution role is created by the same template (<code>DeploymentAutomationLambdaRole</code>) rather than by the next cell. If you run this notebook outside the workshop account, set <code>LAMBDA_ROLE_ARN</code> below to a role that trusts <code>lambda.amazonaws.com</code> and grants <a href="deployment_automation/lambda_role_policy.json">the reference policy</a>.
</div>

## How the function decides

The interesting part of an event-driven deployment is not the deployment, it is deciding *when not to deploy*. The table is the function's contract; the docstring printed below is the same contract from the code's side.

| Question | What the function does |
|---|---|
| What starts it? | The EventBridge rule: `SageMaker Model Package State Change` with `ModelApprovalStatus = Approved` on a group whose name starts with `from-idea-to-prod-model-`. `PendingManualApproval` and `Rejected` never reach the function |
| What does it trust? | `DescribeModelPackage`, not the event. The version has to be `Completed`, `Approved`, in the `production` lifecycle stage (the promotion the governance officer made through the MLflow alias) and carry an inference specification. Anything else is `skipped` |
| First approval | `CreateModel` (a container definition that names the Model Package, nothing else), `CreateEndpointConfig` with data capture, `CreateEndpoint` |
| Later approvals | New `Model` and `EndpointConfig`, then `UpdateEndpoint`: SageMaker provisions the new fleet, health-checks it, moves traffic and releases the old one. The endpoint name and its callers do not change |
| Same version again | `no-op`. It reads the endpoint's current *and pending* endpoint configuration down to the models' `ModelPackageName`, so a duplicate event during a rollout is harmless too |
| Endpoint busy | `Creating` / `Updating` endpoints cannot be updated; the invocation is reported as `deferred`. The next `Approved` event, or a manual invoke, picks the work up |
| Where is the provenance? | Tags on the `Model`, `EndpointConfig` and `Endpoint`: Model Package ARN, group, version, who approved. Nothing is written back to the package, so the function never triggers itself |

In [ ]:
lambda_source_path = Path("deployment_automation/lambda_function.py")
lambda_source = lambda_source_path.read_text()

# Print the module docstring and the function signatures — open the file for the rest.
import ast
module = ast.parse(lambda_source)
print(ast.get_docstring(module))
print("\nFunctions:")
for node in module.body:
    if isinstance(node, ast.FunctionDef):
        print(f"  {node.name}({', '.join(a.arg for a in node.args.args)})")

## Look up the Lambda execution role

The function needs more than the model card automation did, because it creates infrastructure: besides reading the registry it creates models, endpoint configurations and the endpoint, and it has to **pass the SageMaker execution role** to the endpoint so the endpoint can pull the serving image and read the model artifacts from the MLflow artifact store. Each of those is scoped to the exact endpoint name and the resource names derived from it, and the `iam:PassRole` to that one role, for `sagemaker.amazonaws.com` only.

That role already exists. The workshop CloudFormation template `static/cfnSageMakerProjectsRoles.yaml` creates it as `DeploymentAutomationLambdaRole`, for the same reason it creates the model card role in notebook 02 and the two persona roles in notebook 00: **the Studio execution role deliberately has no `iam:CreateRole`**. Least-privilege IAM belongs in the account's infrastructure-as-code, not in a notebook, so this notebook only looks the role up and passes it to Lambda. [`deployment_automation/lambda_role_policy.json`](deployment_automation/lambda_role_policy.json) is a reference copy of the policy, for reading alongside the notebook.

The cell below resolves the role, prints the policy that is actually deployed on it, and checks it against the reference copy, so the blast radius of the function is visible before you create it and a stale stack is caught here rather than at the first approval.

In [ ]:
# Override only if you run this notebook outside the workshop account and want to
# point the function at a role of your own. In the workshop, leave it at None.
LAMBDA_ROLE_ARN = None

# The reference policy, with the placeholders filled in for this account: what the
# function needs. Nothing below applies it; it is the yardstick for the check.
placeholders = {
    "${REGION}": region,
    "${ACCOUNT_ID}": account_id,
    "${MODEL_PACKAGE_GROUP_PREFIX}": model_package_group_prefix,
    "${ENDPOINT_NAME}": deployment_endpoint_name,
    "${EXECUTION_ROLE_ARN}": sm_role,
    "${LAMBDA_FUNCTION_NAME}": deployment_lambda_name,
}
policy_text = Path("deployment_automation/lambda_role_policy.json").read_text()
for placeholder, value in placeholders.items():
    policy_text = policy_text.replace(placeholder, value)
reference_policy = as_iam_policy_document(json.loads(policy_text))

if LAMBDA_ROLE_ARN is not None:
    deployment_lambda_role_arn = LAMBDA_ROLE_ARN
    print(f"Using provided role: {deployment_lambda_role_arn}")
else:
    try:
        deployment_lambda_role_arn = iam_client.get_role(RoleName=deployment_lambda_role_name)["Role"]["Arn"]
        print(f"Using the pre-created role: {deployment_lambda_role_arn}")
    except iam_client.exceptions.NoSuchEntityException:
        raise RuntimeError(
            f"IAM role '{deployment_lambda_role_name}' does not exist.\n"
            "It is provisioned by the workshop CloudFormation stack as the resource\n"
            "DeploymentAutomationLambdaRole in static/cfnSageMakerProjectsRoles.yaml.\n"
            "Update that stack, or set LAMBDA_ROLE_ARN above to a role that trusts\n"
            "lambda.amazonaws.com and grants deployment_automation/lambda_role_policy.json."
        ) from None

def actions_of(policy_document):
    actions = set()
    for statement in policy_document.get("Statement", []):
        if statement.get("Effect") != "Allow":
            continue
        listed = statement.get("Action", [])
        actions.update([listed] if isinstance(listed, str) else listed)
    return actions

def grants(granted, needed):
    """True if `needed` (e.g. sagemaker:CreateModel) is covered by one of `granted`
    (exact, `service:*`, `*`, or a `Prefix*` wildcard). Resource scoping is not checked."""
    for action in granted:
        if action == "*" or action == needed:
            return True
        if action.endswith("*") and needed.startswith(action[:-1]):
            return True
    return False

# Print the policy actually deployed on the role and compare it with the reference.
role_name = deployment_lambda_role_arn.split("/")[-1]
try:
    deployed_actions = set()
    for policy_name in iam_client.list_role_policies(RoleName=role_name)["PolicyNames"]:
        document = iam_client.get_role_policy(RoleName=role_name, PolicyName=policy_name)["PolicyDocument"]
        deployed_actions |= actions_of(document)
        print(f"\nInline policy '{policy_name}' on {role_name}:")
        for statement in document["Statement"]:
            actions = statement["Action"] if isinstance(statement["Action"], list) else [statement["Action"]]
            resources = statement["Resource"] if isinstance(statement["Resource"], list) else [statement["Resource"]]
            print(f"  {statement['Effect']:<6}{', '.join(actions)}")
            for resource in resources:
                print(f"        on {resource}")
    missing = sorted(a for a in actions_of(reference_policy) if not grants(deployed_actions, a))
    if missing:
        print(f"\n[WARN] {role_name} does not grant everything the function needs; its first real")
        print("       approval would fail with AccessDenied. The stack that created the role is")
        print("       probably older than this notebook. Missing actions:")
        for action in missing:
            print(f"         - {action}")
    else:
        print(f"\n{role_name} grants every action in the reference policy.")
except ClientError as e:
    if e.response["Error"]["Code"] not in ("AccessDenied", "AccessDeniedException"):
        raise
    print("\n(not allowed to read this role's policy, so it is not shown or checked here)")

## Deploy the Lambda function

The deployment package is a zip with a single file. Everything that describes *what* to deploy and *where* comes in as environment variables (endpoint name, instance type, execution role, capture prefix, required lifecycle stage), so the same code serves another endpoint or another registry by changing its configuration. Re-running the cell updates the code and configuration of an existing function.

The timeout is generous because the only slow path in the function, replacing a `Failed` endpoint, waits for the deletion to finish before it recreates the endpoint. The normal paths return in seconds: `CreateEndpoint` and `UpdateEndpoint` are asynchronous.

In [ ]:
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(lambda_source_path, arcname="lambda_function.py")
zip_bytes = buffer.getvalue()

lambda_environment = {
    "Variables": {
        "ENDPOINT_NAME": deployment_endpoint_name,
        "EXECUTION_ROLE_ARN": sm_role,
        "INSTANCE_TYPE": inference_instance_type,
        "INSTANCE_COUNT": "1",
        "DATA_CAPTURE_S3_URI": capture_s3_uri,
        "REQUIRED_LIFECYCLE_STAGE": "production",
    }
}

try:
    lambda_client.get_function(FunctionName=deployment_lambda_name)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=deployment_lambda_name)
    lambda_client.update_function_code(FunctionName=deployment_lambda_name, ZipFile=zip_bytes)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=deployment_lambda_name)
    lambda_client.update_function_configuration(
        FunctionName=deployment_lambda_name, Role=deployment_lambda_role_arn,
        Environment=lambda_environment, Timeout=300,
    )
    print(f"Updated function: {deployment_lambda_name}")
except lambda_client.exceptions.ResourceNotFoundException:
    lambda_client.create_function(
        FunctionName=deployment_lambda_name,
        Runtime="python3.12",
        Role=deployment_lambda_role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Description="Deploy an approved from-idea-to-prod Model Package version to the production endpoint",
        Timeout=300,
        MemorySize=256,
        Environment=lambda_environment,
        Tags={"project": project},
    )
    print(f"Created function: {deployment_lambda_name}")

lambda_client.get_waiter("function_active_v2").wait(FunctionName=deployment_lambda_name)
deployment_lambda_arn = lambda_client.get_function(FunctionName=deployment_lambda_name)["Configuration"]["FunctionArn"]
print(f"Function ARN: {deployment_lambda_arn}")

## Create the EventBridge rule

Same event as in notebook 02, [`SageMaker Model Package State Change`](https://docs.aws.amazon.com/sagemaker/latest/dg/automating-sagemaker-with-eventbridge.html#eventbridge-model-package), narrower pattern: besides the group name prefix it requires `ModelApprovalStatus` to be `Approved`. Registrations (`PendingManualApproval`), rejections and metadata updates on unapproved versions never invoke the function. The function still re-checks the registry, because a filter is not a guard: an event can be old by the time it is delivered.

In [ ]:
event_pattern = {
    "source": ["aws.sagemaker"],
    "detail-type": ["SageMaker Model Package State Change"],
    "detail": {
        "ModelPackageGroupName": [{"prefix": model_package_group_prefix}],
        "ModelApprovalStatus": ["Approved"],
    },
}

rule_arn = events_client.put_rule(
    Name=deployment_rule_name,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="Deploy every approved from-idea-to-prod model package version to the production endpoint",
    Tags=[{"Key": "project", "Value": project}],
)["RuleArn"]
print(f"Rule: {rule_arn}")

events_client.put_targets(
    Rule=deployment_rule_name,
    Targets=[{"Id": "deployment-automation", "Arn": deployment_lambda_arn}],
)

try:
    lambda_client.add_permission(
        FunctionName=deployment_lambda_name,
        StatementId="allow-eventbridge-deploy-on-approval-rule",
        Action="lambda:InvokeFunction",
        Principal="events.amazonaws.com",
        SourceArn=rule_arn,
    )
    print("EventBridge is allowed to invoke the function.")
except lambda_client.exceptions.ResourceConflictException:
    print("Invoke permission already present.")

print(json.dumps(event_pattern, indent=2))

---
## Trigger a deployment with a real approval

The rule only fires on *future* events, and the version notebook 03 registered is already `production` / `Approved`, with no endpoint behind it. So the governance officer issues the approval once more: `approve_for_deployment()` is the same `UpdateModelPackage` call notebook 03 made, which SageMaker turns into a real `Model Package State Change` event with `ModelApprovalStatus = Approved`. That is exactly what a fresh approval of a new version looks like to the automation, without waiting for another pipeline run. (Re-run notebook 03's pipeline and approval afterwards, and the endpoint below is updated with that version, through the same path.)

The cell first confirms the version is actually deployable, as the function will, then assumes the governance-officer role for the approval, as in notebooks 01 and 03.

In [ ]:
latest = latest_model_package(sm_client, registered_model_name, prefix=model_package_group_prefix)
model_package_arn = latest["ModelPackageArn"]
model_package_group_name = model_package_arn.split("/")[1]
model_package_version = latest["ModelPackageVersion"]

state = print_lifecycle(sm_client, model_package_arn, "Latest version in the registry:")
if not (state["stage"] == "production" and state["approval_status"] == "Approved"):
    raise RuntimeError(
        f"{model_package_arn} is stage={state['stage']}, approval={state['approval_status']}. "
        "The automation would skip it. Finish the approval gate in notebook 03 first."
    )

def describe_endpoint_or_none(name):
    try:
        return sm_client.describe_endpoint(EndpointName=name)
    except ClientError as e:
        if "Could not find" in e.response["Error"]["Message"]:
            return None
        raise

before = describe_endpoint_or_none(deployment_endpoint_name)
print(f"\nEndpoint {deployment_endpoint_name} before the approval: "
      f"{before['EndpointStatus'] if before else 'does not exist'}")

# Governance officer: approve for deployment. This is the event.
go_sm = assume_persona(governance_officer_role_arn, GOVERNANCE_OFFICER, region=region)
approval_time = time.time()
approve_for_deployment(
    go_sm, model_package_arn, approved_by=GOVERNANCE_OFFICER,
    note=f"Approved for deployment to {deployment_endpoint_name} (notebook 05)",
)
print_lifecycle(sm_client, model_package_arn, "\nAfter the governance officer's approval:")

### Watch the automation react

Within a minute the event reaches the function and the endpoint starts `Creating`. The cell polls for the endpoint to appear and prints the function's log lines for this model package. If nothing has happened after three minutes, it invokes the function directly with an event of the same shape EventBridge delivers, which is the quickest way to tell a delivery problem (rule, target permission) from a function problem (its logs).

In [ ]:
def approval_event(model_package_arn, group_name, version):
    # Same shape as the EventBridge "SageMaker Model Package State Change" event
    return {
        "version": "0",
        "id": "manual-invoke",
        "detail-type": "SageMaker Model Package State Change",
        "source": "aws.sagemaker",
        "account": account_id,
        "region": region,
        "resources": [model_package_arn],
        "detail": {
            "ModelPackageGroupName": group_name,
            "ModelPackageVersion": version,
            "ModelPackageArn": model_package_arn,
            "ModelApprovalStatus": "Approved",
        },
    }

def function_log_lines(since, pattern='"model-package/"'):
    try:
        events = logs_client.filter_log_events(
            logGroupName=f"/aws/lambda/{deployment_lambda_name}",
            startTime=int(since * 1000),
            filterPattern=pattern,
        )["events"]
        return [e["message"].strip()[-200:] for e in events]
    except Exception as e:
        return [f"[WARN] Could not read the function's logs ({e}). Open them in the CloudWatch console instead."]

def wait_for_new_deployment(previous, since, timeout=180):
    """True as soon as the endpoint exists (first approval) or is Updating (later ones)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        current = describe_endpoint_or_none(deployment_endpoint_name)
        if current and (previous is None or current["LastModifiedTime"] > previous["LastModifiedTime"]):
            print(f"  {time.strftime('%H:%M:%S')} | endpoint {deployment_endpoint_name}: {current['EndpointStatus']}")
            return True
        print(f"  {time.strftime('%H:%M:%S')} | waiting for the event to be processed...")
        time.sleep(10)
    return False

if not wait_for_new_deployment(before, approval_time):
    print("\nNo deployment started within 3 minutes. Invoking the function directly with the same event:")
    response = lambda_client.invoke(
        FunctionName=deployment_lambda_name,
        InvocationType="RequestResponse",
        Payload=json.dumps(approval_event(model_package_arn, model_package_group_name, model_package_version)).encode(),
    )
    result = json.loads(response["Payload"].read())
    if response.get("FunctionError"):
        raise RuntimeError(f"The function failed: {result}")
    print(f"  {result['action']}")
    print("  The function works; check the rule's target and the invoke permission in the EventBridge console.")

print("\nFunction logs:")
for line in function_log_lines(approval_time)[-10:]:
    print(f"  {line}")

### Wait for the endpoint

`CreateEndpoint` returns as soon as the request is accepted; provisioning the instance and passing the container's health check takes a few minutes. Then read back what was deployed: the tags the function put on the endpoint, and the chain endpoint → endpoint configuration → model → `ModelPackageName`, which has to be the version the governance officer approved.

In [ ]:
print(f"Polling endpoint: {deployment_endpoint_name}")
terminal_states = {"InService", "Failed"}
while True:
    desc = sm_client.describe_endpoint(EndpointName=deployment_endpoint_name)
    status = desc["EndpointStatus"]
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        f"Endpoint {deployment_endpoint_name} ended with status '{status}': {desc.get('FailureReason')}"
    )

def served_model_package(endpoint_desc):
    config = sm_client.describe_endpoint_config(EndpointConfigName=endpoint_desc["EndpointConfigName"])
    model = sm_client.describe_model(ModelName=config["ProductionVariants"][0]["ModelName"])
    return model["PrimaryContainer"]["ModelPackageName"], config

served_arn, endpoint_config = served_model_package(desc)
endpoint_tags = {t["Key"]: t["Value"] for t in sm_client.list_tags(ResourceArn=desc["EndpointArn"])["Tags"]}
capture = desc["DataCaptureConfig"]

print(f"\nEndpoint {deployment_endpoint_name} is InService.")
print(f"  endpoint config:  {desc['EndpointConfigName']}")
print(f"  serves:           {served_arn}")
print(f"  data capture:     enabled={capture['EnableCapture']} sampling={capture['CurrentSamplingPercentage']}% -> {capture['DestinationS3Uri']}")
print("  tags:")
for key in ("model-package-group", "model-package-version", "approved-by", "deployment-automation"):
    print(f"    {key:<24}{endpoint_tags.get(key, '')}")

assert served_arn == model_package_arn, f"Endpoint serves {served_arn}, expected {model_package_arn}"
print(f"\nThe endpoint serves exactly the version the governance officer approved: v{model_package_version}.")

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/inference-experience/endpoints/details/{}">inference endpoint</a> in the Studio UI</b>'.format(
            domain_id, region, deployment_endpoint_name))
)

---
## Test the endpoint

Nobody wrote a deployment cell, so check that what came up serves the model the registry promised: the held-out test set in one CSV request, as in notebook 04, and the AUC compared to the `TestAuc` the register step recorded on the version.

In [ ]:
os.makedirs("tmp", exist_ok=True)
S3Downloader.download(f"{test_s3_url}/test_x.csv", "tmp")
S3Downloader.download(f"{test_s3_url}/test_y.csv", "tmp")
test_x = pd.read_csv("tmp/test_x.csv", header=None, names=feature_names)
test_y = pd.read_csv("tmp/test_y.csv", header=None).iloc[:, 0].to_numpy()

response = runtime_client.invoke_endpoint(
    EndpointName=deployment_endpoint_name,
    ContentType="text/csv",
    Accept="text/csv",
    Body=test_x.to_csv(index=False, header=False),
)
predictions = np.array(response["Body"].read().decode("utf-8").strip().split("\n"), dtype=float)
endpoint_test_auc = roc_auc_score(test_y, predictions)

recorded = lifecycle_summary(sm_client, model_package_arn)["customer_metadata"].get("TestAuc", "n/a")
print(f"Endpoint test AUC: {endpoint_test_auc:.4f} | TestAuc recorded on v{model_package_version}: {recorded}")
pd.crosstab(index=test_y, columns=np.round(predictions), rownames=["actuals"], colnames=["predictions"])

### Prove idempotency

Run the very same event again. SageMaker delivers duplicate events, and the model card automation from notebook 02 already produced one more `Approved` event when it reconciled the card after the approval above. None of them may create a second endpoint configuration or start an update.

In [ ]:
response = lambda_client.invoke(
    FunctionName=deployment_lambda_name,
    InvocationType="RequestResponse",
    Payload=json.dumps(approval_event(model_package_arn, model_package_group_name, model_package_version)).encode(),
)
result = json.loads(response["Payload"].read())
print(result["action"])
assert result["action"].startswith("no-op"), result

---
## A later approval updates the endpoint in place (optional)

The first approval created the endpoint. Every approval after it takes the other branch of the function: a new `Model` and `EndpointConfig`, then `UpdateEndpoint`. SageMaker provisions the new instances, waits for the container to pass its health check, switches traffic and releases the old instances; the endpoint stays `InService` for callers throughout, and the endpoint name never changes. Because approval is the trigger, **re-approving an earlier version is how you roll back**, with the same audit trail (who, when, why on the version; what is deployed on the endpoint's tags).

Set `RUN_UPDATE_DEMO = True` to see it. The cell needs a second `production` / `Approved` version in the group (the one notebook 01 registered, if you ran it in the same group) and takes two endpoint updates, roughly ten minutes: the governance officer re-approves the previous version, the endpoint switches to it, then re-approves the latest version and the endpoint switches back.

In [ ]:
RUN_UPDATE_DEMO = False

def approved_production_versions(group_name):
    versions = []
    paginator = sm_client.get_paginator("list_model_packages")
    for page in paginator.paginate(ModelPackageGroupName=group_name, SortBy="CreationTime", SortOrder="Descending"):
        for summary in page["ModelPackageSummaryList"]:
            s = lifecycle_summary(sm_client, summary["ModelPackageArn"])
            if s["stage"] == "production" and s["approval_status"] == "Approved":
                versions.append(summary["ModelPackageArn"])
    return versions

def approve_and_wait_for_switch(target_arn, note):
    previous = sm_client.describe_endpoint(EndpointName=deployment_endpoint_name)
    since = time.time()
    approve_for_deployment(go_sm, target_arn, approved_by=GOVERNANCE_OFFICER, note=note)
    print(f"Approved {target_arn.rsplit('/', 1)[-1]} -> waiting for the endpoint to update")
    if not wait_for_new_deployment(previous, since):
        raise RuntimeError("The endpoint did not start updating; check the function's logs.")
    while True:
        current = sm_client.describe_endpoint(EndpointName=deployment_endpoint_name)
        print(f"  {time.strftime('%H:%M:%S')} | status={current['EndpointStatus']}")
        if current["EndpointStatus"] in ("InService", "Failed"):
            break
        time.sleep(30)
    served, _ = served_model_package(current)
    assert served == target_arn, f"Endpoint serves {served}, expected {target_arn}"
    print(f"  now serving {served}")
    for line in function_log_lines(since, pattern='"updated:"'):
        print(f"  {line}")

if not RUN_UPDATE_DEMO:
    print("Skipped. Set RUN_UPDATE_DEMO = True to watch a re-approval update the endpoint in place.")
else:
    others = [arn for arn in approved_production_versions(model_package_group_name) if arn != model_package_arn]
    if not others:
        print(f"Skipped: {model_package_group_name} has no other production/Approved version to switch to.")
    else:
        previous_arn = others[0]
        approve_and_wait_for_switch(previous_arn, "Rollback: re-approved the previous version (notebook 05 demo)")
        approve_and_wait_for_switch(model_package_arn, "Roll forward: re-approved the latest version (notebook 05 demo)")

---
## Summary

In this notebook you:

- **Made the approval the deployment.** An EventBridge rule on `SageMaker Model Package State Change` with `ModelApprovalStatus = Approved` invokes a Lambda function, so the governance officer's `UpdateModelPackage` is the one action that moves a model to production. There is no deployment cell, script or button left to run.
- **Kept the registry as the contract.** The function trusts `DescribeModelPackage`, not the event: `Completed`, `Approved`, `production`, with an inference specification, or it is skipped. The version's inference specification is all it needs to deploy, so the container definition is just the Model Package ARN.
- **Deployed once, updated forever.** One stable endpoint name, created by the first approval and switched blue/green by `UpdateEndpoint` on later ones; re-approving an earlier version rolls back. Data capture is on from the first deployment, so notebook 04's monitoring reads this endpoint unchanged.
- **Made it safe to replay.** Duplicate events, the model card automation's follow-up event and manual re-invokes end in `no-op`; a busy endpoint is `deferred`; and provenance lives on the endpoint's tags, so the function never writes to the package that triggered it.

### Next steps

- **Staged rollouts**: pass a `DeploymentConfig` to `UpdateEndpoint` with a canary or linear `BlueGreenUpdatePolicy` and CloudWatch alarms for `AutoRollbackConfiguration`, so a version that fails in production rolls back without anyone approving anything.
- **Smoke test before traffic**: have the function invoke the new endpoint configuration through a shadow variant, or run the same test set you used above from a second function, before the switch.
- **Deferred deployments**: a `deferred` result today waits for the next event. Put the event on an SQS queue with a visibility timeout, or drive the function from a Step Functions state machine that waits for `InService`, so an approval made during a rollout is never lost.
- **Multi-account**: replace `CreateEndpoint` in this account with a cross-account event bus target or a CodePipeline start in the deployment account, and keep the same registry event as the trigger.

## Clean-up

An `InService` endpoint bills per instance-hour whether or not you call it, so delete it, along with the endpoint configurations and models the automation created for it. The rule is **disabled** rather than deleted, so an approval made later does not quietly create a billing endpoint while the workshop account is idle; set `DELETE_AUTOMATION = True` to remove the rule and the function as well. The execution role is untouched: it belongs to the account's infrastructure stack.

The Model Registry, the MLflow experiment and the S3 data (datasets, capture files) are **kept**.

In [ ]:
DELETE_AUTOMATION = False

if describe_endpoint_or_none(deployment_endpoint_name):
    sm_client.delete_endpoint(EndpointName=deployment_endpoint_name)
    print(f"Deleting endpoint {deployment_endpoint_name}...")
    sm_client.get_waiter("endpoint_deleted").wait(EndpointName=deployment_endpoint_name)
    print(f"Deleted endpoint {deployment_endpoint_name}")
else:
    print(f"Endpoint {deployment_endpoint_name} does not exist")

# Endpoint configurations and models created by the automation for this endpoint.
for label, page_key, name_key, paginator_name, delete in (
    ("endpoint config", "EndpointConfigs", "EndpointConfigName", "list_endpoint_configs",
     lambda n: sm_client.delete_endpoint_config(EndpointConfigName=n)),
    ("model", "Models", "ModelName", "list_models",
     lambda n: sm_client.delete_model(ModelName=n)),
):
    for page in sm_client.get_paginator(paginator_name).paginate(NameContains=deployment_endpoint_name):
        for item in page[page_key]:
            name = item[name_key]
            if name.startswith(f"{deployment_endpoint_name}-v"):
                try:
                    delete(name)
                    print(f"Deleted {label} {name}")
                except ClientError as e:
                    print(f"Could not delete {label} {name}: {e}")

if DELETE_AUTOMATION:
    try:
        events_client.remove_targets(Rule=deployment_rule_name, Ids=["deployment-automation"])
        events_client.delete_rule(Name=deployment_rule_name)
        print(f"Deleted rule {deployment_rule_name}")
    except ClientError as e:
        print(f"Could not delete rule {deployment_rule_name}: {e}")
    try:
        lambda_client.delete_function(FunctionName=deployment_lambda_name)
        print(f"Deleted function {deployment_lambda_name}")
    except ClientError as e:
        print(f"Could not delete function {deployment_lambda_name}: {e}")
else:
    events_client.disable_rule(Name=deployment_rule_name)
    print(f"Disabled rule {deployment_rule_name} (the function {deployment_lambda_name} stays deployed). "
          "Re-enable it, or re-run the rule cell, to turn the automation back on.")

print("\nClean-up complete. The Model Registry, MLflow experiment and S3 data are kept.")

## Additional resources

- [Automating Amazon SageMaker AI with Amazon EventBridge: Model Package State Change](https://docs.aws.amazon.com/sagemaker/latest/dg/automating-sagemaker-with-eventbridge.html#eventbridge-model-package)
- [Deploy a model from the Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-deploy.html)
- [Update models in production: blue/green deployments](https://docs.aws.amazon.com/sagemaker/latest/dg/deployment-guardrails-blue-green.html)
- [UpdateEndpoint API reference](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_UpdateEndpoint.html)
- [Capture data from a real-time endpoint](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture-endpoint.html)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>